# 05 - Clustering y Evaluación de Modelos

Este notebook recibe las componentes principales generadas en `04_pca_and_cluster_profiling.ipynb` y aplica algoritmos de aprendizaje no supervisado (K-Means y validación con DBSCAN/Jerárquico) para agrupar los exoplanetas en familias operativas y evaluar su estabilidad matemática.

## Resumen ejecutivo

- **Entrada**: 731 exoplanetas representados en 4 componentes principales (PC1–PC4).
- **Objetivo**: Identificar el número óptimo de clusters ($K$) y evaluar su cohesión, separación y estabilidad ante variaciones aleatorias.
- **Métricas**: Método del Codo (Inercia/WCSS), Coeficiente de Silhouette y test de estabilidad multisemilla.
- **Entrega**: Modelo K-Means definitivo, métricas de evaluación y dataset etiquetado (`cluster_labels.csv`) para la DEMO en Streamlit.

Cada bloque incluye explicación, código comentado y conclusión. El notebook debe ejecutarse después del 04 y siempre en orden.

## 1. Carga de datos y verificación del contrato con PCA

### Explicación
Cargamos el archivo `pca_scores.csv` entregado por la fase anterior. Verificamos que contenga las 731 observaciones, el identificador único `pl_name` y las 4 componentes principales (`PC1` a `PC4`) que retienen el 90,35 % de la varianza total. 

Separamos la matriz de características $X$ (sobre la que aplicaremos el clustering) del vector de nombres de exoplanetas para mantener la trazabilidad.

In [ ]:
# Carga e inspección de scores de PCA

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples

# Ruta según el contrato acordado en el Notebook 04
PCA_SCORES_PATH = Path("../data/processed/pca/pca_scores.csv")

# 1. Cargar dataset
df_pca = pd.read_csv(PCA_SCORES_PATH)

# 2. Validaciones de contrato
expected_pcs = ['PC1', 'PC2', 'PC3', 'PC4']
assert 'pl_name' in df_pca.columns, "❌ Error: La columna 'pl_name' no está presente."
assert all(pc in df_pca.columns for pc in expected_pcs), "❌ Error: Faltan componentes principales en el archivo."

# 3. Separación de matriz numérica X e identificadores
X_pca = df_pca[expected_pcs].values
planet_names = df_pca['pl_name']

print(f"✅ Contrato verificado con éxito.")
print(f"- Observaciones cargadas: {X_pca.shape[0]}")
print(f"- Dimensiones de entrada (PCs): {X_pca.shape[1]}")
df_pca.head()

✅ Contrato verificado con éxito.
- Observaciones cargadas: 731
- Dimensiones de entrada (PCs): 4


,pl_name,PC1,PC2,PC3,PC4
0,AU Mic b,-0.373950,0.630395,-0.105734,-0.098964
1,AU Mic c,0.611809,1.105543,0.413814,-0.214052
2,BD+05 4868 A b,-1.836488,-0.244548,-0.470042,-0.493883
3,BD-14 3065 b,0.482023,-3.167789,-0.337570,-1.567546
4,DS Tuc A b,-0.071752,-0.144111,-0.639585,-0.115506


> **Conclusión del bloque**: Los datos se han cargado correctamente y cumplen el contrato definido. Disponemos de una matriz $X$ de $731 \times 4$ lista para alimentar los algoritmos de clustering sin necesidad de reescalado adicional.